# 01_baseline_hybrid_rule

## Mục tiêu notebook

Notebook này tạo **baseline dự báo Quantity 56 ngày cho từng SKU** bằng rule đơn giản, không dùng machine learning.

Baseline dùng các thống kê dễ hiểu từ quá khứ:

- Trung bình bán 28 ngày gần nhất.
- Trung bình bán 56 ngày gần nhất.
- Trung bình bán 90 ngày gần nhất.
- Số ngày có bán trong 90 ngày gần nhất.
- Lợi nhuận tổng của từng SKU.
- Trung bình theo cùng thứ trong tuần trong 8 tuần gần nhất.

## Ý tưởng baseline

Mỗi SKU được chia thành nhóm đơn giản:

| Nhóm SKU | Cách hiểu | Cách dự báo |
|---|---|---|
| Không bán gần đây | 90 ngày gần nhất không bán | Dự báo 0 |
| Bán rất thưa | 90 ngày chỉ bán 1–3 ngày | Dự báo `total_qty_90 / 90` |
| SKU top profit | Có bán gần đây và lợi nhuận cao | Kết hợp `MA_28` và `Weekday_Avg_8w` |
| SKU còn lại | Bán tương đối bình thường | Kết hợp `MA_56` và `MA_90` |

## Output cuối cùng

Notebook tạo file:

```text
submission_baseline_hybrid_v01.csv
```

File này có thể dùng để nộp Kaggle nếu pass toàn bộ bước kiểm tra ở cuối notebook.

# Cell 1 — Import thư viện

## Cell này làm gì?

Cell này gọi các thư viện cần dùng trong notebook.

## Giải thích cho người mới

- `pandas`: dùng để đọc file CSV và xử lý dữ liệu dạng bảng.
- `numpy`: dùng cho tính toán số học.
- `Path`: giúp làm việc với đường dẫn file gọn hơn.

Baseline này không dùng model ML, nên không cần `lightgbm`, `xgboost`, `sklearn`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Cell 2 — Khai báo cấu hình chung

## Cell này làm gì?

Cell này khai báo các tham số chính của baseline.

## Vì sao nên để tham số ở đầu notebook?

Vì sau này nếu muốn thử phiên bản khác, team chỉ cần sửa vài dòng ở đây, không phải tìm trong toàn bộ notebook.

Ví dụ:

- Muốn đổi nhóm top profit từ top 20% thành top 10% thì sửa `TOP_PROFIT_QUANTILE`.
- Muốn đổi SKU bán thưa từ 1–3 ngày thành 1–5 ngày thì sửa `SPARSE_SALES_DAYS_THRESHOLD`.
- Muốn đổi tên file output thì sửa `SUBMISSION_FILE_NAME`.

In [2]:
# Số ngày cần dự báo: 28 ngày Public + 28 ngày Private = 56 ngày
FORECAST_HORIZON = 56
PUBLIC_HORIZON = 28

# Các cửa sổ thống kê quá khứ
WINDOW_28 = 28
WINDOW_56 = 56
WINDOW_90 = 90
WEEKDAY_WINDOW = 56  # 8 tuần = 56 ngày

# Rule phân nhóm SKU
SPARSE_SALES_DAYS_THRESHOLD = 3   # SKU bán 1-3 ngày trong 90 ngày được xem là bán rất thưa
TOP_PROFIT_QUANTILE = 0.80        # Top 20% SKU theo profit được xem là nhóm quan trọng

# Tên file output
SUBMISSION_FILE_NAME = "submission_baseline_hybrid_v01.csv"

# Cell 3 — Tự tìm file dữ liệu

## Cell này làm gì?

Cell này tự tìm `train.csv` và `sample_submission.csv` trong một số vị trí phổ biến.

Notebook được viết để dễ chạy trong nhiều môi trường:

- JupyterLab local.
- Anaconda.
- Kaggle Notebook.
- Google Colab nếu upload file vào thư mục làm việc.

## Cách dùng nếu bị lỗi không tìm thấy file

Nếu cell báo lỗi, hãy sửa thủ công 2 dòng sau:

```python
TRAIN_PATH = Path("đường_dẫn_đến_train.csv")
SAMPLE_SUB_PATH = Path("đường_dẫn_đến_sample_submission.csv")
```

Ví dụ nếu file nằm cùng folder với notebook:

```python
TRAIN_PATH = Path("train.csv")
SAMPLE_SUB_PATH = Path("sample_submission.csv")
```

In [3]:
def find_file(file_name):
   
    candidate_dirs = [
        Path("dataset"),          # Quan trọng nhất cho cấu trúc hbaac/dataset
        Path("."),                # Cùng folder với notebook
        Path(".."),               # Folder cha
        Path("../dataset"),       # Dataset nằm ở folder cha
        Path("./data"),
        Path("./data/raw"),
        Path("../data"),
        Path("../data/raw"),
        Path("/mnt/data"),
        Path("/kaggle/input"),
    ]

    for folder in candidate_dirs:
        path = folder / file_name
        if path.exists():
            return path

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob(file_name))
        if len(matches) > 0:
            return matches[0]

    raise FileNotFoundError(
        f"Không tìm thấy {file_name}. "
        f"Hãy kiểm tra file có nằm trong folder dataset/ không."
    )


TRAIN_PATH = find_file("train.csv")
SAMPLE_SUB_PATH = find_file("sample_submission.csv")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = OUTPUT_DIR / SUBMISSION_FILE_NAME

print("TRAIN_PATH:", TRAIN_PATH)
print("SAMPLE_SUB_PATH:", SAMPLE_SUB_PATH)
print("SUBMISSION_PATH:", SUBMISSION_PATH)

TRAIN_PATH: dataset\train.csv
SAMPLE_SUB_PATH: dataset\sample_submission.csv
SUBMISSION_PATH: outputs\submission_baseline_hybrid_v01.csv


# Cell 4 — Đọc dữ liệu

## Cell này làm gì?

Cell này đọc 2 file chính:

- `train.csv`: dữ liệu giao dịch lịch sử.
- `sample_submission.csv`: file mẫu định dạng nộp bài.

## Giải thích code

- `pd.read_csv(...)`: đọc file CSV thành bảng dữ liệu pandas.
- `.shape`: cho biết số dòng và số cột.
- `.head()`: xem 5 dòng đầu tiên.

Nếu đọc file thành công, cần thấy:

- `train` có nhiều dòng giao dịch.
- `sample_submission` có cột `id` và các cột `F1` đến `F28`.

In [4]:
train = pd.read_csv(TRAIN_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train.shape)
print("Sample submission shape:", sample_sub.shape)

print("\nTrain preview:")
display(train.head())

print("\nSample submission preview:")
display(sample_sub.head())

C:\Users\Louis\AppData\Local\Temp\ipykernel_12232\2711858823.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(TRAIN_PATH)


Train shape: (711980, 8)
Sample submission shape: (31944, 29)

Train preview:


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
0,2020-11-17,2000004,SKU-08063,12,242700,2184300,"123559,1",1482709
1,2020-11-17,2000003,SKU-09458,600,"131818,1818",79090909,110000,66000000
2,2020-11-18,2000007,SKU-08062,6,230000,940909,101000,606000
3,2020-11-18,2000006,SKU-09458,240,270000,44181818,110000,26400000
4,2020-11-18,2000005,SKU-09458,240,270000,44181818,110000,26400000



Sample submission preview:


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,SKU-00002_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,SKU-00003_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,SKU-00004_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,SKU-00005_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Cell 5 — Kiểm tra nhanh cấu trúc dữ liệu

## Cell này làm gì?

Cell này kiểm tra dữ liệu trước khi xử lý sâu hơn.

## Vì sao cần làm?

Người mới thường gặp lỗi do:

- Tên cột khác kỳ vọng.
- Cột ngày chưa đúng định dạng.
- Dữ liệu bị thiếu.
- File submission không đúng format.

Cell này giúp phát hiện sớm các lỗi đó.

In [5]:
print("Columns in train:")
print(train.columns.tolist())

print("\nColumns in sample submission:")
print(sample_sub.columns.tolist())

print("\nMissing values in train:")
display(train.isna().sum())

print("\nMissing values in sample submission:")
display(sample_sub.isna().sum())

print("\nNumber of unique SKUs in train:", train["ItemCode"].nunique())
print("Number of rows in sample submission:", len(sample_sub))

Columns in train:
['Date', 'Stt', 'ItemCode', 'Quantity', 'UnitPrice', 'SalesAmount', 'Unit Cost', 'Cost Amount']

Columns in sample submission:
['id', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28']

Missing values in train:


Date           0
Stt            0
ItemCode       0
Quantity       0
UnitPrice      0
SalesAmount    0
Unit Cost      0
Cost Amount    0
dtype: int64


Missing values in sample submission:


id     0
F1     0
F2     0
F3     0
F4     0
F5     0
F6     0
F7     0
F8     0
F9     0
F10    0
F11    0
F12    0
F13    0
F14    0
F15    0
F16    0
F17    0
F18    0
F19    0
F20    0
F21    0
F22    0
F23    0
F24    0
F25    0
F26    0
F27    0
F28    0
dtype: int64


Number of unique SKUs in train: 15972
Number of rows in sample submission: 31944


# Cell 6 — Chuẩn hóa ngày và tạo cột Profit

## Cell này làm gì?

Cell này làm 2 việc:

1. Chuyển cột `Date` sang đúng kiểu ngày tháng.
2. Tạo cột `Profit`.

## Giải thích cho người mới

Dữ liệu ban đầu có thể đọc `Date` dưới dạng text. Nếu để text, Python không thể hiểu đâu là ngày trước, ngày sau.

Dòng này:

```python
train["Date"] = pd.to_datetime(train["Date"])
```

chuyển `Date` thành kiểu ngày tháng.

Dòng này:

```python
train["Profit"] = train["SalesAmount"] - train["Cost Amount"]
```

tính lợi nhuận từng dòng giao dịch.

Metric của bài ưu tiên SKU có profit cao, nên cần biết SKU nào có tổng profit lớn.

In [7]:
train["Date"] = pd.to_datetime(train["Date"])

# Hàm chuyển cột về dạng số.
# Một số cột trong dữ liệu có thể bị đọc thành text do có dấu phẩy, dấu cách hoặc ký tự lạ.
def to_number(series):
    return (
        series
        .astype(str)
        .str.replace(",", ".", regex=False)      # đổi dấu phẩy thập phân thành dấu chấm nếu có
        .str.replace(" ", "", regex=False)       # bỏ khoảng trắng
        .str.replace("\u00a0", "", regex=False)  # bỏ non-breaking space nếu có
        .pipe(pd.to_numeric, errors="coerce")    # chuyển sang số, lỗi thì thành NaN
    )

numeric_cols = ["Quantity", "SalesAmount", "Cost Amount"]

for col in numeric_cols:
    train[col] = to_number(train[col])

# Nếu sau khi convert có NaN, thay bằng 0 để tránh lỗi tính toán
train[numeric_cols] = train[numeric_cols].fillna(0)

# Tạo Profit = doanh thu - chi phí
train["Profit"] = train["SalesAmount"] - train["Cost Amount"]

print("Data types after conversion:")
display(train[numeric_cols + ["Profit"]].dtypes)

print("\nMissing values after conversion:")
display(train[numeric_cols + ["Profit"]].isna().sum())

print("\nDate range:")
print(train["Date"].min(), "→", train["Date"].max())

display(train.head())

Data types after conversion:


Quantity         int64
SalesAmount      int64
Cost Amount    float64
Profit         float64
dtype: object


Missing values after conversion:


Quantity       0
SalesAmount    0
Cost Amount    0
Profit         0
dtype: int64


Date range:
2020-11-17 00:00:00 → 2025-09-05 00:00:00


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount,Profit
0,2020-11-17,2000004,SKU-08063,12,242700,2184300,"123559,1",1482709.0,701591.0
1,2020-11-17,2000003,SKU-09458,600,"131818,1818",79090909,110000,66000000.0,13090909.0
2,2020-11-18,2000007,SKU-08062,6,230000,940909,101000,606000.0,334909.0
3,2020-11-18,2000006,SKU-09458,240,270000,44181818,110000,26400000.0,17781818.0
4,2020-11-18,2000005,SKU-09458,240,270000,44181818,110000,26400000.0,17781818.0


# Cell 7 — Kiểm tra return transaction

## Cell này làm gì?

Cell này kiểm tra các dòng trả hàng.

Trong dữ liệu, dòng trả hàng thường có:

- `Quantity < 0`
- `SalesAmount < 0`
- `Cost Amount < 0`

## Cách xử lý trong baseline này

Để giữ baseline đơn giản:

1. Vẫn cộng các giao dịch theo ngày và SKU.
2. Nếu tổng Quantity của ngày đó bị âm thì đưa về 0.

Lý do:

- Submission cuối cùng yêu cầu forecast không âm.
- Baseline này dự báo nhu cầu bán ra, không dự báo lượng trả hàng.

In [8]:
return_mask = (
    (train["Quantity"] < 0) &
    (train["SalesAmount"] < 0) &
    (train["Cost Amount"] < 0)
)

print("Number of return rows:", int(return_mask.sum()))
print("Return ratio:", round(float(return_mask.mean()), 6))

print("\nReturn transaction preview:")
display(train.loc[return_mask].head())

Number of return rows: 37279
Return ratio: 0.05236

Return transaction preview:


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount,Profit
47,2020-12-09,2000110,SKU-08063,-12,"-209102,25",-2509227,0,-1482709.0,-1026518.0
48,2020-12-09,2000110,SKU-08061,-12,"-231818,1667",-2781818,0,-1897349.0,-884469.0
1185,2021-05-21,2001755,SKU-07770,-18,"-231818,1667",-4172727,-174000,-3132000.0,-1040727.0
1317,2021-06-08,2002157,SKU-02768,-24,-161000,-3864000,-85000,-2040000.0,-1824000.0
1321,2021-06-08,2002162,SKU-10532,-2,-676618,-1353236,"-390065,25",-780131.0,-573105.0


# Cell 8 — Lấy danh sách SKU từ sample_submission

## Cell này làm gì?

Cell này lấy danh sách SKU cần dự báo từ `sample_submission.csv`.

## Vì sao lấy SKU từ sample_submission thay vì train?

Vì Kaggle chấm theo đúng danh sách `id` trong sample submission.

Nếu tự lấy SKU từ train, có thể xảy ra rủi ro:

- Thiếu SKU có trong sample submission.
- Thừa SKU không cần nộp.
- Thứ tự id không khớp.

Do đó, cách an toàn nhất là lấy SKU từ `sample_submission.csv`.

In [9]:
# id có dạng: SKU-00001_validation hoặc SKU-00001_evaluation
sample_ids = sample_sub["id"].astype(str)

# Xóa hậu tố _validation hoặc _evaluation để lấy ItemCode gốc
all_skus = (
    sample_ids
    .str.replace(r"_(validation|evaluation)$", "", regex=True)
    .unique()
)

all_skus = pd.Index(all_skus, name="ItemCode")

print("Number of SKUs from sample submission:", len(all_skus))
print("First 5 SKUs:", list(all_skus[:5]))

Number of SKUs from sample submission: 15972
First 5 SKUs: ['SKU-00001', 'SKU-00002', 'SKU-00003', 'SKU-00004', 'SKU-00005']


# Cell 9 — Gom transaction thành daily sales theo SKU

## Cell này làm gì?

Dữ liệu gốc là transaction-level, nghĩa là một SKU trong một ngày có thể có nhiều dòng giao dịch.

Baseline cần dữ liệu daily-level:

```text
Date | ItemCode | Quantity trong ngày
```

Do đó cần group theo:

```python
["Date", "ItemCode"]
```

và cộng Quantity, SalesAmount, Cost Amount, Profit.

## Lưu ý về Quantity âm

Sau khi cộng theo ngày và SKU, nếu `Quantity` âm thì clip về 0.

```python
daily["Quantity"] = daily["Quantity"].clip(lower=0)
```

Nghĩa là nếu một ngày net quantity bị âm vì return, baseline xem demand ngày đó là 0.

In [10]:
daily = (
    train
    .groupby(["Date", "ItemCode"], as_index=False)
    .agg({
        "Quantity": "sum",
        "SalesAmount": "sum",
        "Cost Amount": "sum",
        "Profit": "sum",
    })
)

# Baseline dự báo nhu cầu bán ra, nên không để Quantity âm
daily["Quantity"] = daily["Quantity"].clip(lower=0)

print("Daily shape:", daily.shape)
display(daily.head())

Daily shape: (507050, 6)


,Date,ItemCode,Quantity,SalesAmount,Cost Amount,Profit
0,2020-11-17,SKU-08063,12,2184300,1482709.0,701591.0
1,2020-11-17,SKU-09458,600,79090909,66000000.0,13090909.0
2,2020-11-18,SKU-08062,6,940909,606000.0,334909.0
3,2020-11-18,SKU-09458,480,88363636,52800000.0,35563636.0
4,2020-11-20,SKU-09458,240,44181818,26400000.0,17781818.0


# Cell 10 — Tạo hàm lấy dữ liệu đủ ngày trong một window gần nhất

## Cell này làm gì?

Cell này tạo một hàm hỗ trợ.

Hàm này sẽ tạo bảng đủ các dòng:

```text
ngày trong window x tất cả SKU cần dự báo
```

Ví dụ với 90 ngày và 15,972 SKU, bảng sẽ có khoảng:

```text
90 x 15,972 = 1,437,480 dòng
```

## Vì sao không tạo full table cho toàn bộ 5 năm?

Nếu tạo full table 5 năm x 15,972 SKU, bảng có thể rất lớn và nặng máy.

Team yếu tech nên chỉ tạo các window cần cho baseline:

- 28 ngày gần nhất.
- 56 ngày gần nhất.
- 90 ngày gần nhất.
- 8 tuần gần nhất.

Cách này nhẹ hơn và dễ chạy hơn.

In [11]:
def make_complete_recent_window(daily_data, all_skus, end_date, window_days):
    """
    Tạo bảng đủ Date x ItemCode trong window gần nhất.
    
    Parameters
    ----------
    daily_data : DataFrame
        Bảng daily đã group theo Date x ItemCode.
    all_skus : Index hoặc list
        Danh sách SKU cần dự báo.
    end_date : Timestamp
        Ngày cuối cùng của train.
    window_days : int
        Số ngày muốn lấy, ví dụ 28, 56, 90.
    
    Returns
    -------
    DataFrame gồm Date, ItemCode, Quantity.
    Nếu một SKU không bán trong ngày nào đó, Quantity = 0.
    """
    start_date = end_date - pd.Timedelta(days=window_days - 1)
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    
    full_index = pd.MultiIndex.from_product(
        [dates, all_skus],
        names=["Date", "ItemCode"]
    )
    
    window = (
        daily_data.loc[
            (daily_data["Date"] >= start_date) &
            (daily_data["Date"] <= end_date),
            ["Date", "ItemCode", "Quantity"]
        ]
        .set_index(["Date", "ItemCode"])
        .reindex(full_index)
        .reset_index()
    )
    
    window["Quantity"] = window["Quantity"].fillna(0)
    return window

# Cell 11 — Xác định ngày cuối train và các ngày tương lai

## Cell này làm gì?

Cell này xác định:

- Ngày cuối cùng trong train.
- 56 ngày tiếp theo cần dự báo.

## Giải thích

Nếu train kết thúc ở ngày `2025-09-05`, thì ngày dự báo đầu tiên là `2025-09-06`.

Dữ liệu cần nộp chia thành:

- 28 ngày đầu: `_validation`.
- 28 ngày sau: `_evaluation`.

In [12]:
last_train_date = daily["Date"].max()

future_dates = pd.date_range(
    start=last_train_date + pd.Timedelta(days=1),
    periods=FORECAST_HORIZON,
    freq="D"
)

print("Last train date:", last_train_date.date())
print("Future start:", future_dates[0].date())
print("Future end:", future_dates[-1].date())
print("Number of future days:", len(future_dates))

Last train date: 2025-09-05
Future start: 2025-09-06
Future end: 2025-10-31
Number of future days: 56


# Cell 12 — Tính thống kê 90 ngày gần nhất

## Cell này làm gì?

Cell này tính 3 thông tin cho từng SKU trong 90 ngày gần nhất:

| Cột | Ý nghĩa |
|---|---|
| `total_qty_90` | Tổng Quantity bán trong 90 ngày gần nhất |
| `sales_days_90` | Số ngày có bán trong 90 ngày gần nhất |
| `ma_90` | Trung bình Quantity mỗi ngày trong 90 ngày gần nhất |

## Vì sao cần 90 ngày?

90 ngày giúp nhận diện SKU bán thưa:

- Nếu `sales_days_90 = 0`: SKU không bán gần đây.
- Nếu `sales_days_90 <= 3`: SKU bán rất thưa.
- Nếu `sales_days_90 >= 4`: SKU còn có hoạt động gần đây.

In [13]:
window_90 = make_complete_recent_window(
    daily_data=daily,
    all_skus=all_skus,
    end_date=last_train_date,
    window_days=WINDOW_90
)

sku_stats_90 = (
    window_90
    .groupby("ItemCode")
    .agg(
        total_qty_90=("Quantity", "sum"),
        sales_days_90=("Quantity", lambda x: int((x > 0).sum())),
        ma_90=("Quantity", "mean"),
    )
    .reset_index()
)

print("sku_stats_90 shape:", sku_stats_90.shape)
display(sku_stats_90.head())

sku_stats_90 shape: (15972, 4)


,ItemCode,total_qty_90,sales_days_90,ma_90
0,SKU-00001,17.0,9,0.188889
1,SKU-00002,599.0,61,6.655556
2,SKU-00003,966.0,59,10.733333
3,SKU-00004,0.0,0,0.000000
4,SKU-00005,0.0,0,0.000000


# Cell 13 — Tính MA_28 và MA_56

## Cell này làm gì?

Cell này tính moving average cho 28 ngày và 56 ngày gần nhất.

## Moving average là gì?

Moving average là trung bình bán trong một khoảng thời gian gần đây.

Ví dụ:

```text
Nếu 56 ngày gần nhất bán tổng 112 sản phẩm:
MA_56 = 112 / 56 = 2 sản phẩm/ngày
```

## Vì sao dùng nhiều window?

- `MA_28`: nhạy hơn với tình hình gần đây.
- `MA_56`: ổn định hơn `MA_28`.
- `MA_90`: ổn định hơn nữa, phù hợp với SKU ít biến động.

In [14]:
def compute_ma(daily_data, all_skus, end_date, window_days):
    """Tính moving average cho từng SKU trong window_days gần nhất."""
    window = make_complete_recent_window(
        daily_data=daily_data,
        all_skus=all_skus,
        end_date=end_date,
        window_days=window_days
    )
    
    ma = (
        window
        .groupby("ItemCode")["Quantity"]
        .mean()
        .reset_index()
        .rename(columns={"Quantity": f"ma_{window_days}"})
    )
    
    return ma

ma_28 = compute_ma(daily, all_skus, last_train_date, WINDOW_28)
ma_56 = compute_ma(daily, all_skus, last_train_date, WINDOW_56)

sku_stats = (
    sku_stats_90
    .merge(ma_28, on="ItemCode", how="left")
    .merge(ma_56, on="ItemCode", how="left")
)

print("sku_stats shape:", sku_stats.shape)
display(sku_stats.head())

sku_stats shape: (15972, 6)


,ItemCode,total_qty_90,sales_days_90,ma_90,ma_28,ma_56
0,SKU-00001,17.0,9,0.188889,0.285714,0.178571
1,SKU-00002,599.0,61,6.655556,3.500000,5.785714
2,SKU-00003,966.0,59,10.733333,12.714286,11.214286
3,SKU-00004,0.0,0,0.000000,0.000000,0.000000
4,SKU-00005,0.0,0,0.000000,0.000000,0.000000


# Cell 14 — Tính profit của từng SKU và đánh dấu top profit

## Cell này làm gì?

Cell này tính tổng profit của từng SKU trên toàn bộ train.

Sau đó đánh dấu SKU thuộc top 20% profit bằng cột:

```text
is_top_profit
```

## Vì sao cần top profit?

Metric của bài chấm theo trọng số profit. SKU profit cao ảnh hưởng điểm nhiều hơn SKU profit thấp.

Vì vậy baseline sẽ xử lý SKU top profit cẩn thận hơn.

## Lưu ý

Nếu SKU có profit âm, phần ranking sẽ xem profit đó là 0 để tránh ưu tiên SKU lỗ.

In [15]:
sku_profit = (
    daily
    .groupby("ItemCode", as_index=False)
    .agg(total_profit_sku=("Profit", "sum"))
)

# Chỉ dùng profit không âm để ranking
sku_profit["profit_for_rank"] = sku_profit["total_profit_sku"].clip(lower=0)

# Rank percentile: SKU càng profit cao thì rank_pct càng gần 1
sku_profit["profit_rank_pct"] = sku_profit["profit_for_rank"].rank(method="average", pct=True)

# Top profit: thuộc top 20% và profit phải dương
sku_profit["is_top_profit"] = (
    (sku_profit["profit_for_rank"] > 0) &
    (sku_profit["profit_rank_pct"] >= TOP_PROFIT_QUANTILE)
)

print("Number of top profit SKUs:", int(sku_profit["is_top_profit"].sum()))
display(sku_profit.sort_values("total_profit_sku", ascending=False).head())

Number of top profit SKUs: 3195


,ItemCode,total_profit_sku,profit_for_rank,profit_rank_pct,is_top_profit
2,SKU-00003,1.670068e+10,1.670068e+10,1.000000,True
1,SKU-00002,8.012686e+09,8.012686e+09,0.999937,True
9197,SKU-09458,2.641121e+09,2.641121e+09,0.999875,True
4,SKU-00005,2.243340e+09,2.243340e+09,0.999812,True
8354,SKU-08589,1.285705e+09,1.285705e+09,0.999750,True


# Cell 15 — Tính Weekday Average trong 8 tuần gần nhất

## Cell này làm gì?

Cell này tính trung bình Quantity theo từng thứ trong tuần cho mỗi SKU.

Ví dụ với một SKU:

| dayofweek | Ý nghĩa | weekday_avg_8w |
|---:|---|---:|
| 0 | Thứ Hai | trung bình các thứ Hai 8 tuần gần nhất |
| 1 | Thứ Ba | trung bình các thứ Ba 8 tuần gần nhất |
| 2 | Thứ Tư | trung bình các thứ Tư 8 tuần gần nhất |

## Vì sao cần weekday average?

Một số SKU có thể bán khác nhau theo ngày trong tuần.

Ví dụ:

- Ngày làm việc bán nhiều hơn cuối tuần.
- Một vài ngày cố định có nhu cầu cao hơn.

Baseline top profit sẽ dùng thông tin này để dự báo tốt hơn.

In [16]:
window_weekday = make_complete_recent_window(
    daily_data=daily,
    all_skus=all_skus,
    end_date=last_train_date,
    window_days=WEEKDAY_WINDOW
)

# pandas quy ước: 0 = Thứ Hai, 6 = Chủ Nhật
window_weekday["dayofweek"] = window_weekday["Date"].dt.dayofweek

weekday_avg = (
    window_weekday
    .groupby(["ItemCode", "dayofweek"], as_index=False)
    .agg(weekday_avg_8w=("Quantity", "mean"))
)

print("weekday_avg shape:", weekday_avg.shape)
display(weekday_avg.head())

weekday_avg shape: (111804, 3)


,ItemCode,dayofweek,weekday_avg_8w
0,SKU-00001,0,0.125
1,SKU-00001,1,0.625
2,SKU-00001,2,0.375
3,SKU-00001,3,0.125
4,SKU-00001,4,0.000


# Cell 16 — Gộp toàn bộ thống kê SKU

## Cell này làm gì?

Cell này gộp các bảng thống kê lại thành một bảng `sku_stats`.

Sau cell này, mỗi SKU có đủ thông tin:

- `total_qty_90`
- `sales_days_90`
- `ma_90`
- `ma_56`
- `ma_28`
- `total_profit_sku`
- `is_top_profit`

## Vì sao cần fillna?

Một số SKU có thể xuất hiện trong sample submission nhưng không xuất hiện trong train hoặc không có dữ liệu trong window gần đây.

Với các SKU đó, ta điền giá trị thiếu bằng 0 để tránh lỗi khi forecast.

In [17]:
sku_stats = sku_stats.merge(
    sku_profit[["ItemCode", "total_profit_sku", "is_top_profit"]],
    on="ItemCode",
    how="left"
)

fill_values = {
    "total_qty_90": 0,
    "sales_days_90": 0,
    "ma_90": 0,
    "ma_28": 0,
    "ma_56": 0,
    "total_profit_sku": 0,
    "is_top_profit": False,
}

sku_stats = sku_stats.fillna(fill_values)
sku_stats["is_top_profit"] = sku_stats["is_top_profit"].astype(bool)

print("Final sku_stats shape:", sku_stats.shape)
display(sku_stats.head())

Final sku_stats shape: (15972, 8)


,ItemCode,total_qty_90,sales_days_90,ma_90,ma_28,ma_56,total_profit_sku,is_top_profit
0,SKU-00001,17.0,9,0.188889,0.285714,0.178571,3.608433e+07,True
1,SKU-00002,599.0,61,6.655556,3.500000,5.785714,8.012686e+09,True
2,SKU-00003,966.0,59,10.733333,12.714286,11.214286,1.670068e+10,True
3,SKU-00004,0.0,0,0.000000,0.000000,0.000000,8.359968e+08,True
4,SKU-00005,0.0,0,0.000000,0.000000,0.000000,2.243340e+09,True


# Cell 17 — Tạo bảng future 56 ngày

## Cell này làm gì?

Cell này tạo bảng tương lai cần dự báo.

Mỗi SKU có 56 dòng, mỗi dòng tương ứng một ngày tương lai.

Ví dụ:

```text
2025-09-06 | SKU-00001
2025-09-07 | SKU-00001
...
```

## Cột `dayofweek`

Cột này dùng để gắn `weekday_avg_8w` phù hợp với từng ngày tương lai.

In [18]:
future_index = pd.MultiIndex.from_product(
    [future_dates, all_skus],
    names=["Date", "ItemCode"]
)

future = future_index.to_frame(index=False)
future["dayofweek"] = future["Date"].dt.dayofweek

print("Future shape:", future.shape)
display(future.head())

Future shape: (894432, 3)


,Date,ItemCode,dayofweek
0,2025-09-06,SKU-00001,5
1,2025-09-06,SKU-00002,5
2,2025-09-06,SKU-00003,5
3,2025-09-06,SKU-00004,5
4,2025-09-06,SKU-00005,5


# Cell 18 — Gắn thống kê quá khứ vào bảng future

## Cell này làm gì?

Cell này gắn các thông tin đã tính từ quá khứ vào từng dòng tương lai.

Sau cell này, mỗi dòng future sẽ biết:

- SKU đó bán bao nhiêu trong 90 ngày gần nhất.
- SKU đó có bán bao nhiêu ngày trong 90 ngày gần nhất.
- SKU đó có phải top profit không.
- Trung bình bán theo cùng thứ trong tuần là bao nhiêu.

Đây là dữ liệu đầu vào để áp rule baseline.

In [19]:
future = future.merge(sku_stats, on="ItemCode", how="left")
future = future.merge(weekday_avg, on=["ItemCode", "dayofweek"], how="left")

# Điền giá trị thiếu bằng 0
numeric_cols_to_fill = [
    "total_qty_90", "sales_days_90", "ma_90", "ma_28", "ma_56",
    "total_profit_sku", "weekday_avg_8w"
]

future[numeric_cols_to_fill] = future[numeric_cols_to_fill].fillna(0)
future["is_top_profit"] = future["is_top_profit"].fillna(False).astype(bool)

print("Future after merge shape:", future.shape)
display(future.head())

Future after merge shape: (894432, 11)


,Date,ItemCode,dayofweek,total_qty_90,sales_days_90,ma_90,ma_28,ma_56,total_profit_sku,is_top_profit,weekday_avg_8w
0,2025-09-06,SKU-00001,5,17.0,9,0.188889,0.285714,0.178571,3.608433e+07,True,0.000
1,2025-09-06,SKU-00002,5,599.0,61,6.655556,3.500000,5.785714,8.012686e+09,True,4.250
2,2025-09-06,SKU-00003,5,966.0,59,10.733333,12.714286,11.214286,1.670068e+10,True,11.125
3,2025-09-06,SKU-00004,5,0.0,0,0.000000,0.000000,0.000000,8.359968e+08,True,0.000
4,2025-09-06,SKU-00005,5,0.0,0,0.000000,0.000000,0.000000,2.243340e+09,True,0.000


# Cell 19 — Tạo forecast bằng Hybrid Rule Baseline

## Cell này làm gì?

Đây là cell quan trọng nhất của notebook.

Cell này áp rule baseline để tạo cột `forecast`.

## Rule baseline

### Rule 1 — SKU không bán gần đây

```text
Nếu sales_days_90 = 0:
    forecast = 0
```

### Rule 2 — SKU bán rất thưa

```text
Nếu sales_days_90 từ 1 đến 3:
    forecast = total_qty_90 / 90
```

### Rule 3 — SKU top profit

```text
Nếu sales_days_90 >= 4 và is_top_profit = True:
    forecast = 0.5 * ma_28 + 0.5 * weekday_avg_8w
```

### Rule 4 — SKU còn lại

```text
Nếu sales_days_90 >= 4 và không thuộc top profit:
    forecast = 0.5 * ma_56 + 0.5 * ma_90
```

## Vì sao dùng vectorized code?

Notebook dùng `np.select` thay vì `.apply(...)` để chạy nhanh hơn và ít lỗi hơn khi dữ liệu lớn.

In [20]:
conditions = [
    # Rule 1: 90 ngày gần nhất không bán
    future["sales_days_90"] == 0,
    
    # Rule 2: SKU bán rất thưa
    (future["sales_days_90"] >= 1) & (future["sales_days_90"] <= SPARSE_SALES_DAYS_THRESHOLD),
    
    # Rule 3: SKU top profit và có bán tương đối
    (future["sales_days_90"] > SPARSE_SALES_DAYS_THRESHOLD) & (future["is_top_profit"]),
    
    # Rule 4: SKU còn lại, có bán tương đối
    (future["sales_days_90"] > SPARSE_SALES_DAYS_THRESHOLD) & (~future["is_top_profit"]),
]

choices = [
    0,
    future["total_qty_90"] / WINDOW_90,
    0.5 * future["ma_28"] + 0.5 * future["weekday_avg_8w"],
    0.5 * future["ma_56"] + 0.5 * future["ma_90"],
]

future["forecast"] = np.select(conditions, choices, default=0)

# Yêu cầu submission: forecast không âm
future["forecast"] = future["forecast"].clip(lower=0)

print("Forecast created.")
display(future[["Date", "ItemCode", "sales_days_90", "is_top_profit", "forecast"]].head())

Forecast created.


,Date,ItemCode,sales_days_90,is_top_profit,forecast
0,2025-09-06,SKU-00001,9,True,0.142857
1,2025-09-06,SKU-00002,61,True,3.875000
2,2025-09-06,SKU-00003,59,True,11.919643
3,2025-09-06,SKU-00004,0,True,0.000000
4,2025-09-06,SKU-00005,0,True,0.000000


# Cell 20 — Kiểm tra nhanh forecast

## Cell này làm gì?

Cell này kiểm tra forecast trước khi tạo file nộp.

Cần đảm bảo:

- Không có giá trị âm.
- Không có giá trị thiếu `NaN`.
- Không có giá trị vô cực `inf`.
- Forecast lớn nhất không quá bất thường.

Nếu forecast lớn bất thường, cần xem lại SKU đó ở bảng top forecast.

In [21]:
print("Forecast summary:")
display(future["forecast"].describe())

print("Number of negative forecasts:", int((future["forecast"] < 0).sum()))
print("Number of NaN forecasts:", int(future["forecast"].isna().sum()))
print("Number of infinite forecasts:", int(np.isinf(future["forecast"].values).sum()))

print("\nTop 20 highest forecasts:")
display(
    future[["Date", "ItemCode", "sales_days_90", "is_top_profit", "ma_28", "ma_56", "ma_90", "weekday_avg_8w", "forecast"]]
    .sort_values("forecast", ascending=False)
    .head(20)
)

Forecast summary:


count    894432.000000
mean          0.082672
std           1.049697
min           0.000000
25%           0.000000
50%           0.000000
75%           0.011111
max         108.125000
Name: forecast, dtype: float64

Number of negative forecasts: 0
Number of NaN forecasts: 0
Number of infinite forecasts: 0

Top 20 highest forecasts:


,Date,ItemCode,sales_days_90,is_top_profit,ma_28,ma_56,ma_90,weekday_avg_8w,forecast
840036,2025-10-28,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
616428,2025-10-14,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
169212,2025-09-16,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
504624,2025-10-07,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
57408,2025-09-09,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
728232,2025-10-21,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
281016,2025-09-23,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
392820,2025-09-30,SKU-09760,69,True,83.5,80.5,98.188889,132.75,108.125
201156,2025-09-18,SKU-09760,69,True,83.5,80.5,98.188889,130.00,106.750
648372,2025-10-16,SKU-09760,69,True,83.5,80.5,98.188889,130.00,106.750


# Cell 21 — Gắn số thứ tự ngày dự báo

## Cell này làm gì?

Cell này tạo cột `horizon`.

`horizon` cho biết đây là ngày dự báo thứ mấy của từng SKU:

```text
horizon = 1  → ngày dự báo đầu tiên
horizon = 2  → ngày dự báo thứ hai
...
horizon = 56 → ngày dự báo thứ 56
```

Cột này cần thiết để chuyển forecast sang các cột `F1` đến `F28`.

In [22]:
future = future.sort_values(["ItemCode", "Date"]).copy()
future["horizon"] = future.groupby("ItemCode").cumcount() + 1

print("Horizon range:", future["horizon"].min(), "→", future["horizon"].max())
display(future[["Date", "ItemCode", "horizon", "forecast"]].head())

Horizon range: 1 → 56


,Date,ItemCode,horizon,forecast
0,2025-09-06,SKU-00001,1,0.142857
15972,2025-09-07,SKU-00001,2,0.142857
31944,2025-09-08,SKU-00001,3,0.205357
47916,2025-09-09,SKU-00001,4,0.455357
63888,2025-09-10,SKU-00001,5,0.330357


# Cell 22 — Tách 28 ngày validation và 28 ngày evaluation

## Cell này làm gì?

Bài yêu cầu 56 ngày dự báo nhưng submission có 2 loại dòng:

| Loại dòng | Ý nghĩa | Ngày forecast |
|---|---|---|
| `_validation` | Public leaderboard | 28 ngày đầu |
| `_evaluation` | Private leaderboard | 28 ngày sau |

Cell này tách bảng `future` thành 2 phần:

- `future_validation`: horizon 1–28.
- `future_evaluation`: horizon 29–56.

Sau đó `future_evaluation` được đổi horizon về 1–28 để khớp cột `F1` đến `F28`.

In [23]:
future_validation = future[future["horizon"] <= PUBLIC_HORIZON].copy()
future_evaluation = future[future["horizon"] > PUBLIC_HORIZON].copy()

# Evaluation là ngày 29-56, nhưng trong file submission vẫn phải ghi vào F1-F28
future_evaluation["horizon"] = future_evaluation["horizon"] - PUBLIC_HORIZON

print("future_validation shape:", future_validation.shape)
print("future_evaluation shape:", future_evaluation.shape)
print("Validation horizon:", future_validation["horizon"].min(), "→", future_validation["horizon"].max())
print("Evaluation horizon:", future_evaluation["horizon"].min(), "→", future_evaluation["horizon"].max())

future_validation shape: (447216, 13)
future_evaluation shape: (447216, 13)
Validation horizon: 1 → 28
Evaluation horizon: 1 → 28


# Cell 23 — Chuyển forecast sang format F1–F28

## Cell này làm gì?

Forecast hiện tại đang ở dạng long format:

```text
Date | ItemCode | horizon | forecast
```

Nhưng Kaggle yêu cầu wide format:

```text
id | F1 | F2 | ... | F28
```

Cell này dùng `.pivot(...)` để chuyển format.

## Kết quả cần có

- `validation_wide`: mỗi SKU có một dòng `_validation`.
- `evaluation_wide`: mỗi SKU có một dòng `_evaluation`.

In [24]:
forecast_cols = [f"F{i}" for i in range(1, PUBLIC_HORIZON + 1)]

validation_wide = (
    future_validation
    .pivot(index="ItemCode", columns="horizon", values="forecast")
    .reset_index()
)

validation_wide.columns = ["ItemCode"] + forecast_cols
validation_wide["id"] = validation_wide["ItemCode"] + "_validation"
validation_wide = validation_wide[["id"] + forecast_cols]


evaluation_wide = (
    future_evaluation
    .pivot(index="ItemCode", columns="horizon", values="forecast")
    .reset_index()
)

evaluation_wide.columns = ["ItemCode"] + forecast_cols
evaluation_wide["id"] = evaluation_wide["ItemCode"] + "_evaluation"
evaluation_wide = evaluation_wide[["id"] + forecast_cols]

print("validation_wide shape:", validation_wide.shape)
print("evaluation_wide shape:", evaluation_wide.shape)

display(validation_wide.head())
display(evaluation_wide.head())

validation_wide shape: (15972, 29)
evaluation_wide shape: (15972, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857,0.142857,0.142857,...,0.330357,0.205357,0.142857,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857
1,SKU-00002_validation,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000,3.875000,1.750000,...,6.437500,5.750000,5.125000,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000
2,SKU-00003_validation,11.919643,6.357143,12.544643,11.919643,15.044643,14.107143,11.857143,11.919643,6.357143,...,15.044643,14.107143,11.857143,11.919643,6.357143,12.544643,11.919643,15.044643,14.107143,11.857143
3,SKU-00004_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_evaluation,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857,0.142857,0.142857,...,0.330357,0.205357,0.142857,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857
1,SKU-00002_evaluation,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000,3.875000,1.750000,...,6.437500,5.750000,5.125000,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000
2,SKU-00003_evaluation,11.919643,6.357143,12.544643,11.919643,15.044643,14.107143,11.857143,11.919643,6.357143,...,15.044643,14.107143,11.857143,11.919643,6.357143,12.544643,11.919643,15.044643,14.107143,11.857143
3,SKU-00004_evaluation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_evaluation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


# Cell 24 — Gộp thành file submission theo đúng sample_submission

## Cell này làm gì?

Cell này gộp 2 bảng:

- `validation_wide`
- `evaluation_wide`

thành một bảng `submission`.

## Vì sao merge với sample_submission?

Ta dùng `sample_sub[["id"]].merge(...)` để đảm bảo:

- Đúng danh sách id.
- Đúng số dòng.
- Đúng thứ tự giống sample submission.

Đây là cách an toàn hơn so với tự concat rồi nộp trực tiếp.

In [25]:
submission_pred = pd.concat(
    [validation_wide, evaluation_wide],
    axis=0,
    ignore_index=True
)

submission = sample_sub[["id"]].merge(submission_pred, on="id", how="left")

print("submission_pred shape:", submission_pred.shape)
print("submission shape:", submission.shape)

display(submission.head())
display(submission.tail())

submission_pred shape: (31944, 29)
submission shape: (31944, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857,0.142857,0.142857,...,0.330357,0.205357,0.142857,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857
1,SKU-00002_validation,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000,3.875000,1.750000,...,6.437500,5.750000,5.125000,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000
2,SKU-00003_validation,11.919643,6.357143,12.544643,11.919643,15.044643,14.107143,11.857143,11.919643,6.357143,...,15.044643,14.107143,11.857143,11.919643,6.357143,12.544643,11.919643,15.044643,14.107143,11.857143
3,SKU-00004_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
31939,SKU-16329_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31940,SKU-16330_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31941,SKU-16331_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31942,SKU-16332_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31943,SKU-16333_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Cell 25 — Kiểm tra file submission trước khi lưu

## Cell này làm gì?

Cell này kiểm tra các điều kiện quan trọng trước khi lưu file.

## Tất cả kết quả nên đạt như sau

```text
Correct columns: True
Correct number of rows: True
Duplicate ids: 0
NaN values: 0
Negative values: 0
Infinite values: 0
```

Nếu có lỗi:

- `NaN values > 0`: có SKU chưa được gán forecast.
- `Negative values > 0`: forecast bị âm, cần clip về 0.
- `Duplicate ids > 0`: file submission bị trùng dòng.
- `Correct number of rows = False`: thiếu hoặc thừa id.

In [26]:
expected_cols = ["id"] + forecast_cols

# Lớp bảo vệ cuối cùng trước khi kiểm tra
submission[forecast_cols] = submission[forecast_cols].fillna(0)
submission[forecast_cols] = submission[forecast_cols].clip(lower=0)

print("Correct columns:", submission.columns.tolist() == expected_cols)
print("Correct number of rows:", submission.shape[0] == sample_sub.shape[0])
print("Duplicate ids:", int(submission["id"].duplicated().sum()))
print("NaN values:", int(submission.isna().sum().sum()))
print("Negative values:", int((submission[forecast_cols] < 0).sum().sum()))
print("Infinite values:", int(np.isinf(submission[forecast_cols].values).sum()))

print("\nSubmission preview:")
display(submission.head())

Correct columns: True
Correct number of rows: True
Duplicate ids: 0
NaN values: 0
Negative values: 0
Infinite values: 0

Submission preview:


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857,0.142857,0.142857,...,0.330357,0.205357,0.142857,0.142857,0.142857,0.205357,0.455357,0.330357,0.205357,0.142857
1,SKU-00002_validation,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000,3.875000,1.750000,...,6.437500,5.750000,5.125000,3.875000,1.750000,5.312500,4.250000,6.437500,5.750000,5.125000
2,SKU-00003_validation,11.919643,6.357143,12.544643,11.919643,15.044643,14.107143,11.857143,11.919643,6.357143,...,15.044643,14.107143,11.857143,11.919643,6.357143,12.544643,11.919643,15.044643,14.107143,11.857143
3,SKU-00004_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


# Cell 26 — Lưu file submission

## Cell này làm gì?

Cell này lưu file CSV cuối cùng.

Sau khi chạy cell này, file sẽ nằm trong folder:

```text
outputs/
```

Tên file:

```text
submission_baseline_hybrid_v01.csv
```

File này có thể dùng để nộp Kaggle nếu các kiểm tra ở cell trước đều đạt.

In [27]:
submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved submission to:", SUBMISSION_PATH)
print("File exists:", SUBMISSION_PATH.exists())

Saved submission to: outputs\submission_baseline_hybrid_v01.csv
File exists: True


# Cell 27 — Ghi log kết quả

## Version

`baseline_hybrid_v01`

## Cách xử lý dữ liệu

- Đọc `train.csv` và `sample_submission.csv`.
- Convert `Date` sang datetime.
- Tạo `Profit = SalesAmount - Cost Amount`.
- Gom transaction thành daily data theo `Date x ItemCode`.
- Nếu Quantity sau khi gom bị âm thì clip về 0.
- Chỉ tạo full daily table cho các window gần nhất để tránh nặng máy.

## Rule forecast

```text
Nếu sales_days_90 = 0:
    forecast = 0

Nếu 1 <= sales_days_90 <= 3:
    forecast = total_qty_90 / 90

Nếu sales_days_90 > 3 và is_top_profit = True:
    forecast = 0.5 * ma_28 + 0.5 * weekday_avg_8w

Nếu sales_days_90 > 3 và is_top_profit = False:
    forecast = 0.5 * ma_56 + 0.5 * ma_90
```

## File output

```text
outputs/submission_baseline_hybrid_v01.csv
```

## Public LB score

Điền sau khi submit:

```text
Public LB = 0.51059
```

## Ghi chú sau khi có score

Điền nhận xét sau khi submit:

- Score tốt hay xấu so với kỳ vọng?
- Có bị forecast quá cao cho SKU bán thưa không?
- Có cần giảm nhóm top profit từ top 20% xuống top 10% không?
- Có cần thử `MA_90` thay vì mix `MA_56 + MA_90` không?